# 02_all_in_one_runner.ipynb

Interaktiver Einstieg, um alle wesentlichen Einstiegspunkte des Projekts (Quicktest, PF/RH-Workflow, Export und Fixture-Generator) aus einer Notebook-Datei zu bedienen. Wähle die Abschnitte, die du benötigst, und passe Parameter direkt in den Codezellen an.

## Was dieses Notebook bietet
- **Quick Regression Test**: Führt denselben kurzen Pytest-Lauf wie `python quickstart_test.py` aus.
- **PF/RH-Workflow**: Ruft `energis.run.rolling_horizon.run_workflow` mit frei anpassbaren Overrides für Run-Mode, Rolling-Horizon-Parameter und Kostenschalter auf.
- **Vollständiger Export**: Nutzt `energis.run.orchestrator.run_all`, um dieselben Configs als Excel/CSV/JSON-Bundle in `exports/` abzulegen.
- **Fixture-Generator**: Optionaler Aufruf von `scripts/generate_regression_fixtures.py`, um Test-Fixtures aus einem (Teil‑)Zeitraum der Excel-Eingabedaten neu zu erzeugen.
- **Modellüberblick**: Kurzreferenz zu Variablen/Constraints in [`docs/methodology.md`](../docs/methodology.md).


## Voraussetzungen
- Abhängigkeiten installieren (wie im README): `pip install pyomo openpyxl pandas numpy pyyaml pytest` plus ggf. Solver (gurobi/hiGHS/glpk).
- Dieses Notebook vom Projektwurzelverzeichnis aus starten, damit relative Pfade zu Configs und Daten stimmen.
- Ein vorhandenes `Import_Data.xlsx` im Repo-Root oder den Pfad in `configs/sites/default.site.yaml` anpassen.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

def find_repo_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / ".git").exists() and (candidate / "energis").exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd())
if REPO_ROOT != Path.cwd():
    os.chdir(REPO_ROOT)
    print(f"Wechsle Arbeitsverzeichnis auf Repo-Root: {REPO_ROOT}")
else:
    print(f"Arbeitsverzeichnis: {REPO_ROOT}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
    print("Repo-Root zum PYTHONPATH hinzugefügt.")


### 1) Konfigurationen festlegen
Passe die Liste an, wenn du andere Site/System/Scenario-Dateien testen möchtest.

Die Reihenfolge der Dateien ist wichtig: spätere YAMLs überschreiben frühere Einträge. 
* `base.yaml` und `tech_catalog.yaml` liefern Defaults,
* die Site- und System-Datei legt Standort- und Anlagenparameter fest,
* die Scenario-Datei steuert Run-Mode, Rolling-Horizon-Settings und Exportpfade.
Wenn du eigene Configs ergänzt, füge sie an passender Stelle in der Liste ein.


In [ ]:
# Standard-Konfigurationsreihenfolge wie in README und Runner.ipynb
default_configs = [
    'configs/base.yaml',
    'configs/tech_catalog.yaml',
    'configs/sites/default.site.yaml',
    'configs/systems/baseline.system.yaml',
    'configs/scenarios/pf_then_rh.workflow.scenario.yaml',
]
config_paths = [str(REPO_ROOT / path) for path in default_configs]
config_paths


### 2) Overrides für Workflow und Kosten (optional)
Setze Werte auf `None`, um Repository-Defaults bzw. die YAML-Werte zu verwenden.

Overrides werden erst nach dem Laden der YAMLs auf den Config-Dict gemerged;
der oben definierte Config-Pfad bleibt unverändert. Du kannst also vorhandene Dateien
weiterhin laden und nur gezielt Felder überschreiben (z. B. nur den Run-Mode oder einzelne Kosten-Schalter).
Falls du später im Notebook andere Overrides testen willst, passt du einfach dieses Dict an
und führst die nachfolgenden Zellen erneut aus.


In [ ]:
overrides = {
    # Beispiel: 'PF_ONLY', 'RH_ONLY' oder 'PF_THEN_RH'
    'scenario': {
        'run_mode': 'PF_THEN_RH',
        'rolling_horizon': {
            # Rolling-Horizon-Fensterlänge in Stunden
            'heat_horizon_hours': 168.0,
            # Commit-Länge in Stunden
            'step_hours': 24.0,
            # Terminal-Policy für Speicher ('free', 'cyclic' oder 'fix')
            'terminal_policy': 'free',
        },
        # Falls du ein gespeichertes PF-Design erzwingen möchtest
        'pf_design_json': None,
        # True erzwingt Design-Fixierung, False deaktiviert sie, None lässt YAML/Einstellungen gelten
        'fix_design': None,
    },
    'costs': {
        # Kostenschalter: auf True/False setzen oder None für YAML-Defaults
        'include_gridcost_in_energy': None,
        'include_demand_charge_in_rh': None,
        'include_co2_cost_in_objective': None,
    },
}

overrides


### 3) PF/RH-Workflow vorbereiten und ausführen
Verwendet dieselbe Logik wie der CLI-Aufruf `python -m energis.run.rolling_horizon` (ohne Exporte).
Die erste Zelle lädt die Configs einmal, merged Overrides, baut optional **vor** dem Solverlauf ein Pyomo-Modell
und exportiert dessen Struktur. Starte den Solver anschließend in der separaten Zelle `run_workflow`,
damit du das Modell vor dem Lösen inspizieren kannst. Passe die Flags `DUMP_MODEL_BEFORE_SOLVE` und `EXPORT_MODEL_JSON` nach Bedarf an.


In [ ]:
from pathlib import Path

from energis.run import orchestrator, rolling_horizon as rh
from notebooks.model_dump_helper import dump_model_structure, export_model_json

# Modell-Dump vor dem Solver aktivieren/deaktivieren
DUMP_MODEL_BEFORE_SOLVE = True
EXPORT_MODEL_JSON = REPO_ROOT / 'notebooks/exports/model_dump.json'

# Configs laden und Overrides zusammenführen
cfg = orchestrator.load_and_merge(config_paths)
if overrides:
    cfg = orchestrator.deep_merge(cfg, overrides)

# Pyomo-Modell auf Basis der (ggf. überschriebenen) Configs erzeugen
if DUMP_MODEL_BEFORE_SOLVE:
    dt_h = float(cfg.get('run', {}).get('dt_h', 1.0))
    table = orchestrator.load_input_excel(
        cfg.get('site', {}).get('input_xlsx', 'Import_Data.xlsx'),
        cfg.get('site', {}),
        dt_hours=dt_h,
    )
    table.ensure_frequency(dt_h)
    table = orchestrator._apply_horizon(table, cfg.get('scenario', {}), dt_h)
    orchestrator._assert_capacity_vs_demand(table, cfg)
    model = orchestrator.build_model(table, cfg, dt_h=dt_h)

    dump_model_structure(model)
    EXPORT_MODEL_JSON.parent.mkdir(parents=True, exist_ok=True)
    export_model_json(model, EXPORT_MODEL_JSON)
    print(f'Pyomo-Modell vor Solverlauf nach {EXPORT_MODEL_JSON} exportiert.')


In [ ]:
# Rolling-Horizon-Workflow starten
workflow_result = rh.run_workflow(config_paths, overrides=overrides)
print(f"Workflow-Schritte: {' -> '.join(workflow_result.plan.steps)}")

if workflow_result.pf_result:
    pf_obj = workflow_result.pf_result.costs.get('objective.OBJ_value_EUR') if workflow_result.pf_result.costs else None
    print(f"PF-Zeitschritte: {len(workflow_result.pf_result.table)}")
    if pf_obj is not None:
        print(f"PF-Objektivwert (EUR): {pf_obj}")

if workflow_result.rh_result:
    print(f"RH-Fenster: {len(workflow_result.rh_result.windows)}")
    print(f"RH-commit Steps: {len(workflow_result.rh_result.table)}")

if workflow_result.design:
    print(f"Design-Heat-Pumps: {', '.join(sorted(workflow_result.design.heat_pumps.keys())) or 'keine'}")
    if workflow_result.design.storage:
        print(f"Storage-Design: {workflow_result.design.storage}")


### 4) Vollständigen Export erzeugen
`run_all` baut das Modell, löst es und schreibt ein Export-Bundle (CSV/Excel/JSON + Plots) in `exports/<timestamp>_<tag>/`.

In [ ]:
from energis.run import orchestrator

export_meta = orchestrator.run_all(config_paths, overrides=overrides)
print('Export-Verzeichnis:', export_meta['outdir'])
print('Scenario-Excel:', export_meta.get('scenario_xlsx'))
print('PF-Design-JSON:', export_meta.get('pf_design_json'))


### 5) Optional: Quick Regression Test
Entspricht `python quickstart_test.py` und nutzt Pytest auf `tests/test_regression.py`. Überspringen, falls kein Solver installiert ist.

In [ ]:
subprocess.run(['python', 'quickstart_test.py'], check=True)


### 6) Optional: Regression-Fixtures neu erzeugen
Schneidet einen Zeitraum aus `Import_Data.xlsx` und legt neue CSV/XLSX-Teilsets plus Referenz-JSON in `tests/data/` ab.

In [ ]:
fixture_cmd = [
    'python',
    'scripts/generate_regression_fixtures.py',
    '--input-xlsx',
    'Import_Data.xlsx',
    '--output-dir',
    'tests/data',
    '--start',
    '2023-01-01T00:00:00',
    '--hours',
    '12',
]
print('Befehl:', ' '.join(fixture_cmd))
# Entferne das Kommentarzeichen, um die Generierung zu starten:
# subprocess.run(fixture_cmd, check=True)
